# SkyGuard AI - Unsupervised ML Training
This notebook trains unsupervised models (Isolation Forest and LSTM Autoencoder) on clean historical data to learn normal weather patterns. It uses a strict chronological split to prevent temporal leakage.

## 1. Setup & Import Verification

In [ ]:
import os
import sys
import subprocess

# 1. Clone the GitHub repository if running in Kaggle
if os.path.exists('/kaggle/working'):
    print("Running in Kaggle environment.")
    repo_url = "https://github.com/D-Tharun/SkyGuard-AI.git"
    repo_dir = "/kaggle/working/SkyGuard-AI"
    
    if not os.path.exists(repo_dir):
        print(f"Cloning repository from {repo_url}...")
        subprocess.run(["git", "clone", repo_url, repo_dir], check=True)
        print("Repository cloned successfully.")
    else:
        print("Repository already exists.")
        
    BASE_DIR = repo_dir
else:
    print("Running locally.")
    BASE_DIR = os.path.abspath('.')

# 2. Add repository root to sys.path
sys.path.append(BASE_DIR)
print(f"\nAdded repository root to sys.path: {BASE_DIR}")

# 3. Verify imports
print("\nVerifying custom module imports...")
try:
    from src.models.iforest_model import MultivariateIForest
    from src.models.lstm_ae_model import TemporalLSTMAE
    from src.qc.physics_qc import PhysicsEngine
    print("✅ Success: All custom modules imported correctly!")
except ModuleNotFoundError as e:
    raise RuntimeError(f"❌ Import verification failed: {e}\nPlease check that the repository is properly cloned.")

## 2. Dataset Discovery & Chronological Split

In [ ]:
import glob
import pandas as pd
import numpy as np

print("Searching for clean historical CSV datasets...")
if os.path.exists('/kaggle/input'):
    search_pattern = '/kaggle/input/**/*2021_2024_hourly.csv'
else:
    search_pattern = os.path.join(BASE_DIR, 'data/raw/open_meteo/*2021_2024_hourly.csv')

csv_files = glob.glob(search_pattern, recursive=True)

# Fail completely if the dataset is missing
if not csv_files or len(csv_files) < 7:
    raise FileNotFoundError(
        f"\n❌ CRITICAL ERROR: Expected at least 7 clean historical CSV files, but found {len(csv_files)}.\n"
        "Please ensure the Kaggle Dataset containing the 2021-2024 hourly open_meteo files is attached to this notebook.\n"
        "Do NOT fall back to benchmark or injected data."
    )

print(f"\nFound {len(csv_files)} clean station files:")
for f in csv_files:
    print(f" - {f}")

print("\nLoading datasets...")
dfs = [pd.read_csv(f) for f in csv_files]
df = pd.concat(dfs, ignore_index=True)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print("\n--- Dataset Summary ---")
print(f"Total files: {len(csv_files)}")
print(f"Total rows: {len(df)}")
print(f"Stations present: {df['station'].unique().tolist()}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Columns: {df.columns.tolist()}")

print("\nRunning PhysicsEngine to add derived features...")
phys = PhysicsEngine()
df = phys.add_derived_features(df)

# Chronological Split (Train: 2021-2023, Val: 2024)
train_df = df[df['timestamp'] < '2024-01-01'].copy()
val_df = df[df['timestamp'] >= '2024-01-01'].copy()

print("\n--- Chronological Split Summary ---")
print(f"Training records (2021-2023): {len(train_df)}")
print(f"Validation records (2024): {len(val_df)}")

## 3. Train Isolation Forest (Unsupervised)

In [ ]:
print("Training Multivariate Isolation Forest...")
iforest = MultivariateIForest(contamination=0.01)
iforest.train(train_df)
print("IForest training complete.")

## 4. Train LSTM Autoencoder (Unsupervised)

In [ ]:
print("Training Temporal LSTM Autoencoder...")
lstm_ae = TemporalLSTMAE(sequence_length=12, latent_dim=16)
history = lstm_ae.train(train_df, epochs=20, batch_size=256)
print("LSTM training complete.")

## 5. Export Artifacts

In [ ]:
out_dir = '/kaggle/working/models' if os.path.exists('/kaggle/working') else 'models'
os.makedirs(out_dir, exist_ok=True)

# Export exactly matching the application's expectation
iforest.save(os.path.join(out_dir, 'iforest_v2.pkl'))
lstm_ae.save(os.path.join(out_dir, 'lstm_ae_v2'))

print(f"Models exported successfully to {out_dir}")